# 실습 1-4. /chat/stream 구현

## 실습 목표

이번 실습은 앞 세 실습에서 익힌 OpenAI Streaming, async generator, StreamingResponse 세 조각을 합쳐 **정식 `/chat/stream` 엔드포인트**를 완성하는 단계입니다. Pydantic 요청 모델로 입력을 검증하고, generator + StreamingResponse로 토큰 단위 응답을 흘려보내는 표준 패턴을 손에 익힙니다. 이번 실습이 끝나면 같은 패턴을 재활용해 system 프롬프트 고정 버전과 예외/종료 마커가 들어간 안전 버전으로 확장할 수 있게 됩니다.

주요 작업은 다음과 같습니다.

- Pydantic `ChatRequest` 모델로 입력 검증 (필수 `message`, 선택 `model`, `temperature`)
- generator에서 OpenAI 스트림을 받아 `delta.content`를 `yield` → `StreamingResponse`로 흘리기
- 응답 가공이 일어나는 자리(generator 안 `yield` 한 줄) 손에 익히기
- 같은 패턴을 재활용해 system 프롬프트 고정 버전, 예외/종료 마커가 들어간 안전 버전으로 확장

## 1. 클라이언트 준비

3일차에서 쓰던 방식 그대로, MLAPI 접속 정보를 `.env`에서 읽습니다. `.env`에 값이 없으면 입력 프롬프트로 받아 `.env` 파일에 저장하고, server.py는 이 `.env`에서 키를 읽습니다. 키가 코드나 server.py에 직접 박히지 않습니다.

In [20]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

# server.py가 별도 프로세스에서 읽을 수 있도록 .env 파일에 저장
Path(".env").write_text(
    f"MLAPI_BASE_URL={base_url}\n"
    f"MLAPI_API_KEY={api_key}\n"
    f"MLAPI_MODEL={model_name}\n",
    encoding="utf-8",
)
Path(".gitignore").write_text(".env\n__pycache__/\n", encoding="utf-8")

PORT = 8000                  # 서버 포트. 충돌나면 8001 등으로 변경
SERVER_BASE = f"http://localhost:{PORT}"

print("모델명:", model_name, "/ PORT:", PORT)
print("server base:", SERVER_BASE)
print(".env 저장 완료 — server.py가 이 파일에서 키를 읽습니다")

모델명: openai/gpt-5.4 / PORT: 8000
server base: http://localhost:8000
.env 저장 완료 — server.py가 이 파일에서 키를 읽습니다


## 2. server.py 자동 생성

아래 셀을 실행하면 노트북과 같은 폴더에 **server.py** 가 만들어집니다. 이 파일에는 다음이 들어 있습니다.

- Pydantic 요청 모델 `ChatRequest` (필수 `message`, 선택 `model`, `temperature`)
- `/health` : 살아있음 확인
- `/chat/stream` : 정식 스트리밍 챗 엔드포인트 (POST, generator + StreamingResponse)

이 셀은 **파일을 쓰기만 합니다.** 서버는 다음 단계에서 터미널로 직접 띄웁니다.

In [2]:
from pathlib import Path

# server.py 베이스 — ChatRequest · /health · /chat/stream. TODO 셀이 이 뒤에 엔드포인트를 덧붙임
server_base = '''import os
from typing import Optional
from dotenv import load_dotenv
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from openai import OpenAI

load_dotenv()
BASE_URL = os.getenv("MLAPI_BASE_URL")
API_KEY  = os.getenv("MLAPI_API_KEY")
MODEL    = os.getenv("MLAPI_MODEL", "openai/gpt-4o-mini")

if not BASE_URL:
    raise RuntimeError("MLAPI_BASE_URL이 설정되어 있지 않습니다. .env 파일을 확인하세요.")
if not API_KEY:
    raise RuntimeError("MLAPI_API_KEY가 설정되어 있지 않습니다. .env 파일을 확인하세요.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)


app = FastAPI()


class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, description="사용자 질문")
    model: Optional[str] = Field(None, description="사용할 모델, 미지정 시 서버 기본값")
    temperature: Optional[float] = Field(None, ge=0.0, le=2.0)


@app.get("/health")
def health():
    return {"ok": True}


@app.post("/chat/stream")
def chat_stream(req: ChatRequest):
    def gen():
        stream = client.chat.completions.create(
            model=req.model or MODEL,
            messages=[{"role": "user", "content": req.message}],
            temperature=req.temperature if req.temperature is not None else 1.0,
            stream=True,
        )
        for chunk in stream:
            if not chunk.choices:
                continue
            piece = getattr(chunk.choices[0].delta, "content", None)
            if not piece:
                continue
            yield piece
    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(server_base, encoding="utf-8")
print("server.py 작성 완료 (ChatRequest · /health · /chat/stream)")

server.py 작성 완료 (ChatRequest · /health · /chat/stream)


## 3. 서버 띄우기

터미널에서 아래 셀이 출력하는 명령어를 그대로 붙여넣어 실행하세요.

- `--reload` 덕분에 server.py 저장 시 자동 재시작
- 중지: 터미널에서 `Ctrl + C`
- 포트 충돌 시 2 단계 `PORT` 만 바꾸고 2~4 단계 셀 재실행
- 노트북 셀로 서버를 띄우려 하지 말 것

In [3]:
print(f"uvicorn server:app --reload --port {PORT}")
print()
print("서버 주소:", SERVER_BASE)

uvicorn server:app --reload --port 8000

서버 주소: http://localhost:8000


In [4]:
import httpx

r = httpx.get(f"{SERVER_BASE}/health", timeout=5.0)
print(r.status_code, r.json())

200 {'ok': True}


## 4. /chat/stream 기본형 호출

server.py 의 본문은 이렇게 생겼습니다.

```python
class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1)
    model: Optional[str] = None
    temperature: Optional[float] = Field(None, ge=0.0, le=2.0)

@app.post("/chat/stream")
def chat_stream(req: ChatRequest):
    def gen():
        stream = client.chat.completions.create(
            model=req.model or MODEL,
            messages=[{"role": "user", "content": req.message}],
            temperature=req.temperature if req.temperature is not None else 1.0,
            stream=True,
        )
        for chunk in stream:
            if not chunk.choices:
                continue
            piece = getattr(chunk.choices[0].delta, "content", None)
            if not piece:
                continue
            yield piece
    return StreamingResponse(gen(), media_type="text/plain")
```

조각이 어디서 왔는지 짚어 봅니다.

- `ChatRequest` : 3 일차에서 익힌 Pydantic 모델. `message` 가 필수, 나머지는 선택.
- `client.chat.completions.create(..., stream=True)` : 1-1 의 OpenAI 스트리밍.
- chunk 루프 + 빈 chunk 가드 + `yield piece` : 1-2 의 변환 generator 패턴.
- `StreamingResponse(gen(), media_type="text/plain")` : 1-3 의 StreamingResponse.

노트북에서 `httpx` 로 POST 호출합니다.

In [ ]:
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream",
        json={"message": "파이썬 데코레이터를 간단하게 설명해줘"},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

## 5. 응답 가공 위치 손에 익히기

5 단계의 `/chat/stream` 안에서 "가공이 들어갈 자리" 가 어디인지 직접 손으로 확인합니다.

실습 1-2 에서 본 "소비 → 가공 → 재방출" 패턴이 `/chat/stream` 의 어디에 박혀 있는지 보면:

- 원본 흐름: `client.chat.completions.create(..., stream=True)` 가 chunk 를 한 조각씩 흘림
- 소비·가공: `for chunk in stream:` 루프 안에서 `chunk.choices` 확인 → `delta.content` 만 뽑음
- 재방출: `yield piece` 로 본문에 흘려보냄

이 "재방출" 한 줄을 잠깐 바꿔서 응답이 진짜 변하는지 직접 확인합니다.

**해볼 것**
1. server.py 를 열어 `/chat/stream` 의 `yield piece` 한 줄을 다음으로 바꾸고 저장한다.
   ```python
   yield piece.replace(" ", "·")
   ```
2. 터미널의 uvicorn 로그에 `Reloading...` 이 떠 자동 반영되는지 확인.
3. 아래 호출 셀을 다시 실행. 응답의 공백이 모두 가운데점(`·`)으로 보이면 가공이 그 자리에 들어간 것.
4. 확인했으면 server.py 의 그 줄을 다시 `yield piece` 로 **원복**하고 저장. 뒤의 TODO 에서 깨끗한 상태가 필요합니다.

In [8]:
# 6 단계 호출 (변환 적용 전후 모두 이 셀로 확인)
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream",
        json={"message": "FastAPI 가 스트리밍 응답을 만드는 방식을 한 문장으로 설명해줘"},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

status: 200
FastAPI는·보통·`StreamingResponse`를·사용해·제너레이터나·async·이터레이터가·데이터를·조금씩·생성할·때마다·이를·즉시·클라이언트로·전송하는·방식으로·스트리밍·응답을·만듭니다.
--- end ---


# TODO

지금까지 본 패턴을 직접 손에 익히기 위한 과제입니다.

## TODO 1. 여러 질문 던지기

server.py 는 그대로 두고, 노트북에서 여러 질문을 같은 엔드포인트로 던져 응답이 어떻게 흐르는지 비교하세요.

- 질문 3 개 이상을 리스트로 만들기
- 각 질문마다 `/chat/stream` 에 POST → 응답 본문을 토큰 단위로 출력
- 질문/응답 사이를 구분선으로 명확히 구분

**검증 포인트**
- 모든 질문에 대해 토큰 단위로 흐르는 응답이 보이는가
- 짧은 질문/긴 질문/한국어/영어 등을 섞었을 때 모두 동작하는가

In [ ]:
# TODO 1
import httpx

# TODO 1-1: 질문 3개 이상 (짧은/긴/한국어/영어 섞기)
questions = [
    "who is the best progamer in the world?",
    "한국의 수도는 어디인가요?",
    "인셉션에서 팽이를 토템으로 사용해 지금 있는 공간이 현실인지 꿈속인지 구분하는데 만약에 인셉션처럼 \
    꿈속에서 또 꿈속으로 들어가는 상황이 무한히 반복된다면 어떻게 구분할 수 있을까요?",
]

with httpx.Client(timeout=60.0) as c:
    for i, q in enumerate(questions, 1):
        print(f"\n===== [{i}] {q} =====")
        # TODO 1-2: q 를 /chat/stream 에 POST 하고 토큰을 흘려 출력 (5단계 호출 셀 참고)
        with c.stream(
            "POST",
            f"{SERVER_BASE}/chat/stream",
            json={"message": q},
        ) as r:
            print("status:", r.status_code)
            for piece in r.iter_text():
                print(piece, end="", flush=True)
print("\n--- end ---")

## TODO 2. /chat/stream/v2 - system 프롬프트 고정

server.py 에 새 엔드포인트 `/chat/stream/v2` 를 추가합니다.

- POST, 요청 모델은 `ChatRequest` 그대로 재사용
- `messages` 에 system 프롬프트를 한 줄 고정으로 넣고, 그 뒤에 사용자 `message` 를 user 로
- `model`·`temperature` 는 요청 값 우선, 없으면 서버 기본값
- 빈 chunk 가드 + `media_type="text/plain"` 으로 `StreamingResponse` 반환

아래 셀에서 `todo2` 의 빈칸을 채우고 실행하면 server.py 가 갱신됩니다. `--reload` 자동 반영 → 호출 셀로 확인.

In [11]:
# TODO 2 — /chat/stream/v2 엔드포인트
todo2 = '''

SYSTEM_PROMPT_V2 = "너는 항상 두 문장 이내로 짧게 답하는 한국어 챗봇이야."

@app.post("/chat/stream/v2")
def chat_stream_v2(req: ChatRequest):
    def gen():
        stream = client.chat.completions.create(
            model=req.model or MODEL,

            # TODO 2: messages 에 system(SYSTEM_PROMPT_V2) + user(req.message) 두 줄
            messages=[
                None
            ],

            temperature=req.temperature if req.temperature is not None else 0.7,
            stream=True,
        )
        for chunk in stream:
            if not chunk.choices:
                continue
            piece = getattr(chunk.choices[0].delta, "content", None)
            if not piece:
                continue
            yield piece
    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(server_base + todo2, encoding="utf-8")
print("server.py 갱신 — /chat/stream/v2 추가")

server.py 갱신 — /chat/stream/v2 추가


In [16]:
# TODO 2 호출: 길게 답할 만한 질문을 던져도 두 문장 이내로 짧게 답하는지 확인
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/v2",
        json={"message": "tell me about shortpoint and merite of FastAPI  "},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

status: 200
FastAPI의 **장점(merits)**: 매우 빠른 성능, `async` 지원, 자동 API 문서(Swagger/OpenAPI), 타입 힌트 기반 검증, 개발 생산성이 높다는 점입니다.  
**단점(short points)**: Django보다 내장 기능이 적고, 초보자에겐 비동기 개념이 어렵고, 대규모 프로젝트에서는 구조를 직접 잘 설계해야 합니다.
--- end ---


## TODO 3. /chat/stream/safe - 예외 처리 + 종료 마커

server.py 에 새 엔드포인트 `/chat/stream/safe` 를 추가합니다. `/chat/stream` 에 안전장치를 더한 형태입니다.

- POST, 요청 모델은 `ChatRequest` 재사용
- generator 내부를 `try/except` 로 감쌈
- 정상 종료: 스트림이 끝까지 흐르면 마지막에 `\n[DONE]` 한 줄
- 예외 시: `\n[ERROR] {타입}: {메시지}` 한 줄 흘리고 종료 — 서버 프로세스는 죽지 않음
- 빈 chunk 가드 + `media_type="text/plain"` 으로 `StreamingResponse` 반환

아래 셀에서 `todo3` 의 ①②③(`# TODO 3-1·3-2·3-3`) 빈칸을 채우고 실행하세요.

**검증** — 정상: 본문 끝에 `[DONE]`. 오류: server.py 의 `/chat/stream/safe` `gen()` 첫 줄에 임시로 `raise RuntimeError("test")` 를 넣어 저장 → 본문에 `[ERROR] RuntimeError: test` 가 흐르고 uvicorn 은 살아있으면 정답. 확인 후 임시 줄은 지움.

In [17]:
# TODO 3 — /chat/stream/safe 엔드포인트
todo3 = '''

@app.post("/chat/stream/safe")
def chat_stream_safe(req: ChatRequest):
    def gen():
        try:
            # TODO 3-1: /chat/stream 과 동일한 스트리밍 (create stream=True + 빈 chunk 가드 + piece yield)
            None

            # TODO 3-2: 정상 종료 마커 — "\\n[DONE]" 한 줄 yield
            None

        except Exception as e:
            # TODO 3-3: 예외 마커 — "\\n[ERROR] {타입}: {메시지}" 한 줄 yield
            None
            
    return StreamingResponse(gen(), media_type="text/plain")
'''

Path("server.py").write_text(server_base + todo2 + todo3, encoding="utf-8")
print("server.py 갱신 — /chat/stream/safe 추가")

server.py 갱신 — /chat/stream/safe 추가


In [19]:
# TODO 3 호출: 정상 케이스
import httpx

with httpx.Client(timeout=60.0) as c:
    with c.stream(
        "POST",
        f"{SERVER_BASE}/chat/stream/safe",
        json={"message": "파이썬 데코레이터를 한 문장으로 설명해줘"},
    ) as r:
        print("status:", r.status_code)
        for piece in r.iter_text():
            print(piece, end="", flush=True)
print("\n--- end ---")

status: 200
파이썬 데코레이터는 함수를 직접 수정하지 않고도 전후에 추가 동작을 붙여 기능을 확장하는 문법입니다.
[DONE]
--- end ---


오류 케이스 검증은 server.py 안의 `/chat/stream/safe` 의 `gen()` 첫 줄에 임시로 `raise RuntimeError("test")` 를 넣고 저장한 뒤(자동 리로드됨), 위 호출 셀을 한 번 더 실행해 보세요. 응답 본문에 `[ERROR] RuntimeError: test` 한 줄이 흐르고, 서버 터미널은 그대로 살아 있어야 합니다. 확인 후 임시 `raise` 줄은 지워주세요.

## 정리

- `/chat/stream` 의 본체는 결국 세 조각의 합: OpenAI `stream=True` (3-1) + 변환 generator 패턴 (3-2) + StreamingResponse (3-3).
- 입력은 Pydantic 요청 모델로 받는다. `message` 는 필수, `model`/`temperature` 는 선택으로 두면 같은 엔드포인트로 다양한 호출 패턴을 흡수할 수 있다.
- system 프롬프트 같은 "고정 정책" 은 v2 처럼 별도 엔드포인트나 분기점으로 명시하는 것이 안전하다. 한 엔드포인트에 너무 많은 옵션을 박지 않는다.
- 스트리밍 응답에서의 예외는 HTTP status 로 표현하기 어렵다. 이미 200 으로 본문을 흘리기 시작한 상태이기 때문. 그래서 generator 내부 try/except 로 잡아서 본문 끝에 `[ERROR] ...` 한 줄을 흘려보내는 패턴이 가장 단순하다. 정상 종료를 알리는 `[DONE]` 마커도 같은 자리에 둔다.
- 구조는 server.py(FastAPI 앱) + 터미널(uvicorn `--reload`) + 노트북(httpx 클라이언트) 세 갈래. 서버 코드를 수정하면 터미널 로그에 reload 가 뜨고, 노트북에서는 호출만 다시 하면 된다.